In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

print("🔄 Loading lightweight text encoder (this takes just a few seconds)...")
# Using a small, fast model to generate 384-dimensional text embeddings
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

# 1. LOAD THE EXTENDED DATASET FROM THE KAGGLE PATH
# Replacing the hardcoded 5-company dictionary with your real 300-sample CSV file
csv_path = "/kaggle/input/datasets/chakrabhuanavdeva/fake-company-data/train.csv"
df = pd.read_csv(csv_path)

# 2. HELPER FUNCTION TO CONVERT TEXT TO TENSOR EMBEDDINGS
def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    # Use Mean Pooling to get a single vector for the sentence
    return outputs.last_hidden_state.mean(dim=1).squeeze().tolist()

print(f"🧠 Converting {len(df)} text descriptions into mathematical embeddings...")
# Assumes your train.csv contains a 'text' column that provides corporate contexts
df['embedding'] = df['company'].apply(get_embedding)

# 3. SAVE IT FOR YOUR LTN PROJECT
# Keeping the exact same file target for continuity down the pipeline
df.to_pickle("finance_five_companies.pkl")
print("✅ Done! Dataset 'finance_five_companies.pkl' is ready for your LTN logic rules!")

# Let's peek at the newly extended dataset schema
print("\n📊 Your Generated Test Dataset Summary:")
print(df[['company', 'is_profitable', 'has_high_debt', 'has_strong_cash_flow', 'has_good_management', 'has_regulatory_issues', 'recommend_buy']].head())

🔄 Loading lightweight text encoder (this takes just a few seconds)...


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🧠 Converting 300 text descriptions into mathematical embeddings...
✅ Done! Dataset 'finance_five_companies.pkl' is ready for your LTN logic rules!

📊 Your Generated Test Dataset Summary:
              company  is_profitable  has_high_debt  has_strong_cash_flow  \
0          Theta Corp              1              0                     0   
1       Pinnacle Tech              1              0                     0   
2  Aether Enterprises              1              1                     1   
3     Alpha Solutions              1              0                     0   
4       Zeta Dynamics              0              1                     1   

   has_good_management  has_regulatory_issues  recommend_buy  
0                    0                      0              0  
1                    0                      0              0  
2                    0                      0              0  
3                    1                      0              1  
4                    1             

In [2]:
!pip install LTNtorch

In [3]:
import torch
import pandas as pd
import ltn

# Dynamically handle device allocation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Using Device: {device.type.upper()}")

# 1. LOAD THE DATASET
df = pd.read_pickle("finance_five_companies.pkl")

# Convert lists of embeddings and all feature labels into standard PyTorch tensors
embeddings = torch.tensor(df['embedding'].tolist(), dtype=torch.float32, device=device)
labels_profitable = torch.tensor(df['is_profitable'].tolist(), dtype=torch.float32, device=device)
labels_high_debt = torch.tensor(df['has_high_debt'].tolist(), dtype=torch.float32, device=device)
labels_cash_flow = torch.tensor(df['has_strong_cash_flow'].tolist(), dtype=torch.float32, device=device)
labels_management = torch.tensor(df['has_good_management'].tolist(), dtype=torch.float32, device=device)
labels_regulatory = torch.tensor(df['has_regulatory_issues'].tolist(), dtype=torch.float32, device=device)
labels_buy = torch.tensor(df['recommend_buy'].tolist(), dtype=torch.float32, device=device)

# 2. DEFINE THE NEURAL NETWORKS (PREDICATES)
class FinanceModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = torch.nn.Sequential(
            torch.nn.Linear(384, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 1),
            torch.nn.Sigmoid() 
        )
    def forward(self, x):
        return self.layer(x)

# Create independent models for every financial label
model_profit = FinanceModel().to(device)
model_debt = FinanceModel().to(device)
model_cash = FinanceModel().to(device)
model_mgmt = FinanceModel().to(device)
model_reg = FinanceModel().to(device)
model_buy = FinanceModel().to(device)

# Wrap them into LTN Predicates
IsProfitable = ltn.Predicate(model_profit)
HasHighDebt = ltn.Predicate(model_debt)
HasStrongCashFlow = ltn.Predicate(model_cash)
HasGoodManagement = ltn.Predicate(model_mgmt)
HasRegulatoryIssues = ltn.Predicate(model_reg)
RecommendBuy = ltn.Predicate(model_buy)

# 3. SET UP FUZZY LOGIC OPERATORS
Not = ltn.Connective(ltn.fuzzy_ops.NotStandard())
And = ltn.Connective(ltn.fuzzy_ops.AndProd())
Or = ltn.Connective(ltn.fuzzy_ops.OrMax())
Implies = ltn.Connective(ltn.fuzzy_ops.ImpliesReichenbach())
Forall = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")

# 4. SETUP OPTIMIZER
all_parameters = (
    list(model_profit.parameters()) + list(model_debt.parameters()) + 
    list(model_cash.parameters()) + list(model_mgmt.parameters()) + 
    list(model_reg.parameters()) + list(model_buy.parameters())
)
optimizer = torch.optim.Adam(all_parameters, lr=0.01)

# 5. TRAINING LOOP
print(f"\n🚀 Starting training on {len(df)} samples using {device.type.upper()}...")
for epoch in range(150):
    optimizer.zero_grad()
    
    x_all = ltn.Variable("x_all", embeddings)
    x_prof = ltn.Variable("x_prof", embeddings[labels_profitable == 1])
    x_not_prof = ltn.Variable("x_not_prof", embeddings[labels_profitable == 0])
    x_debt = ltn.Variable("x_debt", embeddings[labels_high_debt == 1])
    x_not_debt = ltn.Variable("x_not_debt", embeddings[labels_high_debt == 0])
    x_cash = ltn.Variable("x_cash", embeddings[labels_cash_flow == 1])
    x_not_cash = ltn.Variable("x_not_cash", embeddings[labels_cash_flow == 0])
    x_mgmt = ltn.Variable("x_mgmt", embeddings[labels_management == 1])
    x_not_mgmt = ltn.Variable("x_not_mgmt", embeddings[labels_management == 0])
    x_reg = ltn.Variable("x_reg", embeddings[labels_regulatory == 1])
    x_not_reg = ltn.Variable("x_not_reg", embeddings[labels_regulatory == 0])
    x_buy = ltn.Variable("x_buy", embeddings[labels_buy == 1])
    x_not_buy = ltn.Variable("x_not_buy", embeddings[labels_buy == 0])
    
    # --- DEFINE LOGIC RULES (THE KNOWLEDGE BASE) ---
    rules = []
    
    # Data Supervision Rules
    rules.append(Forall(x_prof, IsProfitable(x_prof)))
    rules.append(Forall(x_not_prof, Not(IsProfitable(x_not_prof))))
    rules.append(Forall(x_debt, HasHighDebt(x_debt)))
    rules.append(Forall(x_not_debt, Not(HasHighDebt(x_not_debt))))
    rules.append(Forall(x_cash, HasStrongCashFlow(x_cash)))
    rules.append(Forall(x_not_cash, Not(HasStrongCashFlow(x_not_cash))))
    rules.append(Forall(x_mgmt, HasGoodManagement(x_mgmt)))
    rules.append(Forall(x_not_mgmt, Not(HasGoodManagement(x_not_mgmt))))
    rules.append(Forall(x_reg, HasRegulatoryIssues(x_reg)))
    rules.append(Forall(x_not_reg, Not(HasRegulatoryIssues(x_not_reg))))
    rules.append(Forall(x_buy, RecommendBuy(x_buy)))
    rules.append(Forall(x_not_buy, Not(RecommendBuy(x_not_buy))))
    
    # Domain Knowledge Rules (Target Condition Mapping)
    has_strength = Or(HasStrongCashFlow(x_all), HasGoodManagement(x_all))
    clean_baseline = And(IsProfitable(x_all), And(Not(HasHighDebt(x_all)), Not(HasRegulatoryIssues(x_all))))
    target_condition = And(clean_baseline, has_strength)
    
    # Target implies condition & condition implies target
    rules.append(Forall(x_all, Implies(RecommendBuy(x_all), target_condition)))
    rules.append(Forall(x_all, Implies(target_condition, RecommendBuy(x_all))))
    
    # Combine all rules using logical 'AND'
    knowledge_base = rules[0]
    for rule in rules[1:]:
        knowledge_base = And(knowledge_base, rule)
        
    sat_level = knowledge_base.value
    loss = 1.0 - sat_level
    
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1:3d} | Satisfaction Level: {sat_level.item():.4f} | Loss: {loss.item():.4f}")

print("\n🎉 Training complete!")

🖥️ Using Device: CPU

🚀 Starting training on 300 samples using CPU...
Epoch  30 | Satisfaction Level: 0.4565 | Loss: 0.5435
Epoch  60 | Satisfaction Level: 0.7749 | Loss: 0.2251
Epoch  90 | Satisfaction Level: 0.7760 | Loss: 0.2240
Epoch 120 | Satisfaction Level: 0.7762 | Loss: 0.2238
Epoch 150 | Satisfaction Level: 0.7763 | Loss: 0.2237

🎉 Training complete!


In [5]:
import torch
import pandas as pd
import ltn
from transformers import AutoTokenizer, AutoModel

# 1. SETUP ENVIRONMENT AND LOAD TRAINED MODEL
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the text encoder to embed test company names
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
encoder_model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2").to(device)

# Switch only the target model to evaluation mode
model_buy.eval()

# 2. READ AND MERGE KAGGLE TEST AND SOLUTION DATASETS
df_test = pd.read_csv("/kaggle/input/datasets/chakrabhuanavdeva/fake-company-data/test.csv")
df_sol = pd.read_csv("/kaggle/input/datasets/chakrabhuanavdeva/fake-company-data/solution.csv")

# Merge solution target column into the test dataset on company name
df_eval = pd.merge(df_test, df_sol[['company', 'recommend_buy']], on='company', how='inner')

# 3. GENERATE TEXT EMBEDDINGS FOR TEST COMPANIES
def get_eval_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = encoder_model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().tolist()

print(f"🧠 Generating true semantic embeddings for {len(df_eval)} evaluation companies...")
df_eval['embedding'] = df_eval['company'].apply(get_eval_embedding)

# Convert evaluation vectors into standard PyTorch tensor on active device
test_embeddings = torch.tensor(df_eval['embedding'].tolist(), dtype=torch.float32, device=device)

# 4. RUN INFERENCE ONLY FOR THE TARGET METRIC
with torch.no_grad():
    pred_buy_fuzzy = RecommendBuy.model(test_embeddings).squeeze(-1)

# 5. POST-PROCESS CONTINUOUS FUZZY VALUES INTO BINARY LABELS
df_eval['pred_recommend_buy_prob'] = pred_buy_fuzzy.cpu().numpy()

threshold = 0.5
df_eval['pred_recommend_buy'] = (df_eval['pred_recommend_buy_prob'] >= threshold).astype(int)

# 6. DISPLAY VERIFICATION SPREAD
print("\n🔮 Evaluation Results (First 10 Rows):")
cols_to_show = [
    'company', 
    'is_profitable', 
    'has_high_debt', 
    'recommend_buy', 
    'pred_recommend_buy'
]
print(df_eval[cols_to_show].head(10).to_string(index=False))

# 7. PRINT OVERALL TARGET ACCURACY METRIC
print("\n📊 Accuracy Summary Metrics:")
acc = (df_eval['recommend_buy'] == df_eval['pred_recommend_buy']).mean() * 100
print(f"🎯 Accuracy for {'recommend_buy':<25}: {acc:.2f}%")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🧠 Generating true semantic embeddings for 100 evaluation companies...

🔮 Evaluation Results (First 10 Rows):
           company  is_profitable  has_high_debt  recommend_buy  pred_recommend_buy
     Pinnacle Labs              0              0              0                   0
    Delta Partners              1              0              1                   0
          Chi Tech              1              1              0                   1
    Sigma Dynamics              1              1              0                   0
Aether Enterprises              1              1              0                   0
        Nu Startup              1              1              0                   1
     Nexus Systems              1              1              0                   1
      Rho Ventures              1              0              1                   0
   Phi Enterprises              1              0              1                   0
      Rho Partners              1              0   